# EEG 3A Processing


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get update -qq
!apt-get install -y -qq git-annex datalad
!pip install -q mne pymatreader h5py pandas


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import sys
import importlib

code_folder = '/content/drive/MyDrive/EEG'

if code_folder not in sys.path:
    sys.path.append(code_folder)

import process_raw_data
import openneuro_loader

importlib.reload(process_raw_data)
importlib.reload(openneuro_loader)

from process_raw_data import process_all_data
from openneuro_loader import download_openneuro_data


In [ ]:
SOURCE = 'openneuro'

OPENNEURO_SUBJECTS = ['032']


In [ ]:
if SOURCE == 'drive':
    directory_name = '/content/drive/MyDrive/EEG Shared/RawData/Kosachenko'
    extension = 'set'
    win_len = 2
    freq = 256
    output_directory = '/content/drive/MyDrive/EEG Shared/ProcessedData/Kosachenko_2s'

elif SOURCE == 'openneuro':
    directory_name = download_openneuro_data(
        dataset_id='ds003838',
        snapshot='1.0.6',
        subjects=OPENNEURO_SUBJECTS,
        local_root='/content',
    )

    extension = 'set'
    win_len = 2
    freq = 256
    output_directory = '/content/drive/MyDrive/EEG Shared/ProcessedData/Kosachenko_2s'

else:
    raise ValueError("SOURCE must be 'drive' or 'openneuro'.")

print('Directory:', directory_name)
print('Extension:', extension)
print('Window length:', win_len)
print('Frequency:', freq)


$ datalad clone https://github.com/OpenNeuroDatasets/ds003838.git /content/ds003838
$ git fetch --tags
$ git checkout 1.0.6
EEG subjects available: 65
Subjects selected: 1

$ datalad get -r sub-032/eeg

Downloaded EEG .set files: 130
Event TSV files available: 380
Directory: /content/ds003838
Extension: set
Window length: 2
Frequency: 256


In [ ]:
data = process_all_data(
    directory_name,
    extension,
    win_len,
    freq,
    output_directory=output_directory,
)


WORKFLOW 3A - RAW EEG PREPROCESSING
Files found: 2
Window length: 2 s
Sampling frequency: 256 Hz
Rows per epoch: 24
Expected model sample: 24 x 95
Aggregation: RMS = sqrt(mean(x^2))
Output directory: /content/drive/MyDrive/EEG Shared/ProcessedData/Kosachenko_2s

[1/2]

Processing: sub-032_task-memory_eeg.set


/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(


  workload digit events: 1458
  features per row: 19 channels x 5 bands = 95
  model sample shape: 24 x 95
  epochs: 1458 | output rows: 34992
  saved -> /content/drive/MyDrive/EEG Shared/ProcessedData/Kosachenko_2s/sub-032_task-memory_eeg.csv

[2/2]

Processing: sub-032_task-rest_eeg.set


/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(
/usr/local/lib/python3.12/dist-packages/pymatreader/utils.py:225: UserWarning: Complex objects (like classes) are not supported. They are imported on a best effort base but your mileage will vary.
  warn(


  no matching workload events; using continuous fallback
  features per row: 19 channels x 5 bands = 95
  model sample shape: 24 x 95
  epochs: 113 | output rows: 2712
  saved -> /content/drive/MyDrive/EEG Shared/ProcessedData/Kosachenko_2s/sub-032_task-rest_eeg.csv

FINAL CHECK
Rows: 37704
Epochs / model samples: 1571
Rows per epoch: 24
Feature columns: 95
Model sample shape: (24, 95)
Target values present: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Files skipped: 0


In [ ]:
print('Shape:', data.shape)
print('Rows:', len(data))
print('Columns:', len(data.columns))
print('Epochs:', data['epoch_uid'].nunique())
print('Targets:', sorted(data['target'].dropna().astype(int).unique().tolist()))
display(data.head(30))


Shape: (37704, 112)
Rows: 37704
Columns: 112
Epochs: 1571
Targets: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


,dataset,subject_id,recording_id,label,condition,source_file,absolute_load,target,event_index,epoch_id,...,Cz_Gamma,C4_Gamma,T8_Gamma,P7_Gamma,P3_Gamma,Pz_Gamma,P4_Gamma,P8_Gamma,O1_Gamma,O2_Gamma
0,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000006,0.000011,0.000007,0.000006,0.000006,0.000005,0.000010,0.000008,0.000008
1,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000006,0.000006,0.000006,0.000006,0.000005,0.000005,0.000006,0.000006,0.000007
2,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000007,0.000011,0.000006,0.000005,0.000005,0.000006,0.000008,0.000006,0.000006
3,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000005,0.000009,0.000004,0.000005,0.000005,0.000006,0.000006,0.000006,0.000005
4,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000004,0.000006,0.000010,0.000008,0.000005,0.000005,0.000004,0.000014,0.000009,0.000010
5,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000005,0.000017,0.000005,0.000006,0.000005,0.000006,0.000018,0.000010,0.000012
6,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000004,0.000005,0.000011,0.000004,0.000005,0.000005,0.000006,0.000006,0.000006,0.000006
7,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000004,0.000005,0.000012,0.000005,0.000005,0.000004,0.000006,0.000012,0.000005,0.000005
8,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000007,0.000008,0.000008,0.000005,0.000005,0.000005,0.000017,0.000009,0.000009
9,Kosachenko,032,sub-032_task-memory_eeg,memory,control,sub-032_task-memory_eeg.set,13,1,1,0,...,0.000005,0.000006,0.000013,0.000006,0.000004,0.000005,0.000005,0.000011,0.000007,0.000007
